# Remote Inference — Cognitive Similarity

Runs TRIBE v2 inference on IBC stimuli and saves raw tensors to Google Drive.
All cells are safe to re-run — already-processed stimuli are skipped automatically.

**Runtime:** Google Colab Pro with A100 GPU.

## Cell 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone/pull the git repo
import os
REPO_URL = "https://github.com/gdavid7/cogsimexperiment.git"
REPO_DIR = "/content/cogsimexperiment"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Install dependencies matching tribev2's requirements
# tribev2 requires: numpy==2.2.6, torch>=2.5.1,<2.7
# Strategy: Let tribev2 install its own dependencies, then install our extras

print("Installing tribev2 with its dependencies...")
!pip install -q git+https://github.com/facebookresearch/tribev2.git

print("Installing additional dependencies...")
!pip install -q nilearn scikit-learn hypothesis pandas transformers huggingface-hub jedi

print("\n" + "="*60)
print("✓ Installation complete!")
print("="*60)
print("\nNOTE: The runtime will now restart automatically.")
print("After restart, continue with Cell 1b.\n")

# Use IPython's restart mechanism instead of os.kill for cleaner restart
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 1b — Login (run after runtime restarts)

In [ ]:
# After the runtime restarts, re-run drive mount and login
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/cogsimexperiment"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gdavid7/cogsimexperiment.git {REPO_DIR}

# Verify installation
import numpy as np
import torch
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# HuggingFace login (for gated Llama 3.2 access used by TRIBE v2)
from huggingface_hub import login
login()  # will prompt for token

## Cell 2 — Load Models

In [ ]:
import sys
import os

# Define paths
REPO_DIR = "/content/cogsimexperiment"
CACHE_FOLDER = "/content/drive/MyDrive/cognitive-similarity-cache"

# Add repo to path
sys.path.insert(0, REPO_DIR)

# Import TRIBE v2
from tribev2 import TribeModel

# Create cache folder if it doesn't exist
os.makedirs(CACHE_FOLDER, exist_ok=True)

print("Loading cortical model...")
# Cortical model (default)
cortical_model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device="cuda"  # Explicitly use GPU
)

print("Loading subcortical model...")
# Subcortical model
subcortical_model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device="cuda",  # Explicitly use GPU
    config_update={
        "data.neuro.projection": {
            "name": "MaskProjector",
            "mask": "subcortical",
            "=replace=": True
        },
        "data.neuro.fwhm": 6.0
    }
)

print("✓ Models loaded successfully.")

## Cell 3 — Clone IBC Protocols, Preconvert Stimuli, Build Manifest

Clones [`individual-brain-charting/public_protocols`](https://github.com/individual-brain-charting/public_protocols) and picks hand-curated exemplars from `FaceBody/stimuli/` — the IBC adaptation of the Stigliani fLoc localizer (11 category subfolders × 144 `.jpg`).

**Why we convert JPGs to MP4s.** `TribeModel.get_events_dataframe(video_path=...)` hard-gates on `.avi/.mkv/.mov/.mp4/.webm` and raises `ValueError` on any other suffix. TRIBEv2.pdf §5.9 describes the intended workaround: *"All images are presented to TRIBE in a randomized order, for one second every eight seconds (transforming them to static videos)."* The library also ships an undocumented helper (`CreateVideosFromImages` in `tribev2/eventstransforms.py`, `fps=10`, `libx264`, no audio); we replicate the same output with `ffmpeg` and cache the MP4s under `DRIVE_CACHE/stimulus_videos/`.

**Duration = 1 s (matches the paper).** Caveat: TRIBE's internal V-JEPA2 extractor has `clip_duration=4 s`, so a 1-s input partially pads V-JEPA2's window. The paper used 1 s anyway; if peak responses look degraded, extend the duration as a first debug step.

Only the 8 exemplars needed for `ValidationSuite`'s **Visual System** checks are materialised. The other 5 validation checks (auditory / language / MT+) need stimuli that are not shipped in this repo — see the medium-term note in the chat.

| stimulus_id | IBC category | source file |
|---|---|---|
| `face_01` / `face_02` | `adult` (face-selective via FFA) | `adult-1.jpg`, `adult-2.jpg` |
| `place_01` / `place_02` | `house` (place-selective via PPA) | `house-1.jpg`, `house-2.jpg` |
| `body_01` / `body_02` | `body` (body-selective via EBA) | `body-1.jpg`, `body-2.jpg` |
| `written_character_01` / `02` | `word` (VWFA-selective) | `word-1.jpg`, `word-2.jpg` |

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# Ensure repo is in path
sys.path.insert(0, REPO_DIR)

from cognitive_similarity.cache import ResponseCache
from cognitive_similarity.models import Stimulus

DRIVE_CACHE = Path(CACHE_FOLDER)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

# Clone IBC public_protocols repository
IBC_PROTOCOLS_DIR = Path("/content/ibc_public_protocols")
if not IBC_PROTOCOLS_DIR.exists():
    print("Cloning IBC public_protocols repository...")
    !git clone https://github.com/individual-brain-charting/public_protocols.git {IBC_PROTOCOLS_DIR}
else:
    print("✓ IBC public_protocols already cloned")

# FaceBody/stimuli/ has 11 category subfolders (adult, body, car, child,
# corridor, house, instrument, limb, number, scrambled, word), each with
# 144 images named <category>-<N>.jpg.
STIMULI_ROOT = IBC_PROTOCOLS_DIR / "FaceBody" / "stimuli"
if not STIMULI_ROOT.exists():
    raise FileNotFoundError(
        f"Expected FaceBody stimuli at {STIMULI_ROOT}. "
        "The IBC repository layout may have changed."
    )

# TRIBE v2's get_events_dataframe(video_path=...) accepts only
# .avi/.mkv/.mov/.mp4/.webm — feeding a .jpg raises ValueError.
#
# Per TRIBEv2.pdf §5.9 (Visual experiments): "All images are presented to
# TRIBE in a randomized order, for one second every eight seconds
# (transforming them to static videos)." The library actually ships a
# (not-publicly-documented) helper for this — CreateVideosFromImages in
# tribev2/eventstransforms.py — which uses moviepy ImageClip with
# fps=10, codec=libx264, audio=False. We do the same with ffmpeg
# (preinstalled on Colab) so the output parameters match what TRIBE v2's
# own helper would produce.
#
# Note: TRIBE v2's internal V-JEPA2 extractor uses clip_duration=4 s
# (tribev2/grids/defaults.py), so a 1-s video means V-JEPA2's window is
# partially padded. The paper used 1 s regardless; if peak responses
# look degraded, try extending the duration.
VIDEO_DIR = DRIVE_CACHE / "stimulus_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

def image_to_static_video(
    image_path: Path,
    video_path: Path,
    duration_s: float = 1.0,
    fps: int = 10,
) -> None:
    """Convert a still image to a static MP4 matching tribev2's own defaults."""
    if video_path.exists():
        return
    subprocess.run(
        [
            "ffmpeg",
            "-loop", "1",
            "-i", str(image_path),
            "-t", str(duration_s),
            "-r", str(fps),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-an",                # no audio track (matches CreateVideosFromImages)
            "-y",
            "-loglevel", "error",
            str(video_path),
        ],
        check=True,
    )

# Hand-picked exemplars for the 8 IDs ValidationSuite expects.
# Two from the same IBC subcategory per label, so sim(A_01, A_02) is
# maximally high and the sim(A,A) > sim(A,B) orderings are as clean as
# possible.
EXEMPLAR_MAP = {
    "face_01":              STIMULI_ROOT / "adult" / "adult-1.jpg",
    "face_02":              STIMULI_ROOT / "adult" / "adult-2.jpg",
    "place_01":             STIMULI_ROOT / "house" / "house-1.jpg",
    "place_02":             STIMULI_ROOT / "house" / "house-2.jpg",
    "body_01":              STIMULI_ROOT / "body"  / "body-1.jpg",
    "body_02":              STIMULI_ROOT / "body"  / "body-2.jpg",
    "written_character_01": STIMULI_ROOT / "word"  / "word-1.jpg",
    "written_character_02": STIMULI_ROOT / "word"  / "word-2.jpg",
}

missing = [sid for sid, p in EXEMPLAR_MAP.items() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing expected IBC files: {missing}. "
        "Check that FaceBody/stimuli/ has the adult/house/body/word subfolders."
    )

# Convert each .jpg to a 1-second static MP4 (cached across runs).
print("\nConverting JPGs to 1s static MP4s (ffmpeg, fps=10)...")
video_paths = {}
for stimulus_id, jpg_path in EXEMPLAR_MAP.items():
    mp4_path = VIDEO_DIR / f"{stimulus_id}.mp4"
    image_to_static_video(jpg_path, mp4_path)
    video_paths[stimulus_id] = mp4_path
    print(f"  ✓ {stimulus_id:<25} → {mp4_path.name}")

# Build manifest. The .mp4 is the stimulus input TRIBE v2 actually sees,
# so we hash the .mp4 bytes; we also record the originating .jpg for
# provenance.
cache = ResponseCache(str(DRIVE_CACHE))
manifest = []
for stimulus_id, mp4_path in video_paths.items():
    category = stimulus_id.rsplit("_", 1)[0]
    stim = Stimulus(video_path=str(mp4_path), stimulus_id=stimulus_id)
    content_hash = cache._content_hash(stim)
    jpg_path = EXEMPLAR_MAP[stimulus_id]
    manifest.append({
        "stimulus_id": stimulus_id,
        "category": category,
        "modality": "image",
        "local_path": str(mp4_path),
        "content_hash": content_hash,
        "tensor_dir": f"tensors/{content_hash}",
        "ibc_source": f"FaceBody/stimuli/{jpg_path.parent.name}/{jpg_path.name}",
    })

manifest_path = DRIVE_CACHE / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f"\n=== Manifest summary ===")
print(f"Stimuli: {len(manifest)} (ValidationSuite visual-system exemplars)")
for entry in manifest:
    print(f"  {entry['stimulus_id']:<25} ← {entry['ibc_source']}")
print(f"\nManifest saved to: {manifest_path}")
print(f"Static MP4s cached at: {VIDEO_DIR}")

## Cell 3b — Smoke Test

Runs ONE stimulus end-to-end before the full inference loop. Confirms:
1. The 1-s static MP4 produced by Cell 3 is accepted by TRIBE v2's `get_events_dataframe(video_path=...)`.
2. The cortical and subcortical models return tensors of the expected shape.

Nothing is written to cache. If this cell fails, Cell 4 would fail identically but much more slowly — stop here and diagnose.

In [ ]:
from cognitive_similarity.stimulus_runner import StimulusRunner
from cognitive_similarity.models import Stimulus

runner = StimulusRunner(cortical_model, subcortical_model)

smoke_entry = manifest[0]  # face_01 → 1s static MP4 from adult-1.jpg
smoke_stim = Stimulus(
    video_path=smoke_entry["local_path"],
    stimulus_id=smoke_entry["stimulus_id"],
)

print(f"Smoke test: {smoke_stim.stimulus_id} ← {smoke_entry['ibc_source']}")
print(f"Running TRIBE v2 on {smoke_stim.video_path} ...\n")

smoke_response = runner.run(smoke_stim)

print(f"✓ cortical    shape={smoke_response.cortical.shape}    dtype={smoke_response.cortical.dtype}    (expected (T, 20484))")
print(f"✓ subcortical shape={smoke_response.subcortical.shape} dtype={smoke_response.subcortical.dtype} (expected (T, 8802))")

assert smoke_response.cortical.shape[1] == 20484, (
    f"Unexpected cortical width: {smoke_response.cortical.shape}"
)
assert smoke_response.subcortical.shape[1] == 8802, (
    f"Unexpected subcortical width: {smoke_response.subcortical.shape}"
)
assert smoke_response.cortical.shape[0] == smoke_response.subcortical.shape[0], (
    "T must match between cortical and subcortical"
)

print(f"\n✓ Smoke test passed (T = {smoke_response.cortical.shape[0]} timepoint(s)). Proceed to Cell 4.")

## Cell 4 — Resume-from-Checkpoint Inference Loop

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from cognitive_similarity.stimulus_runner import StimulusRunner
from cognitive_similarity.models import Stimulus
import numpy as np
from pathlib import Path

runner = StimulusRunner(cortical_model, subcortical_model)

processed = skipped = failed = 0

print(f"\n=== Processing {len(manifest)} stimuli ===\n")

for i, entry in enumerate(manifest, 1):
    stim = Stimulus(video_path=entry["local_path"], stimulus_id=entry["stimulus_id"])
    tensor_dir = DRIVE_CACHE / entry["tensor_dir"]
    cortical_path = tensor_dir / "raw_cortical.npy"
    subcortical_path = tensor_dir / "raw_subcortical.npy"

    # Skip if already cached
    if cortical_path.exists() and subcortical_path.exists():
        print(f"[{i}/{len(manifest)}] ✓ {entry['stimulus_id']:30s} (cached)")
        skipped += 1
        continue

    print(f"[{i}/{len(manifest)}] Processing {entry['stimulus_id']}...")
    
    try:
        # Run inference
        brain_response = runner.run(stim)

        # Verify shapes
        assert brain_response.cortical.shape[1] == 20484, f"Expected cortical shape (T, 20484), got {brain_response.cortical.shape}"
        assert brain_response.subcortical.shape[1] == 8802, f"Expected subcortical shape (T, 8802), got {brain_response.subcortical.shape}"

        # Save tensors
        tensor_dir.mkdir(parents=True, exist_ok=True)
        np.save(cortical_path, brain_response.cortical)
        np.save(subcortical_path, brain_response.subcortical)
        
        print(f"  ✓ Saved: cortical={brain_response.cortical.shape}, subcortical={brain_response.subcortical.shape}")
        processed += 1
        
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        failed += 1
        continue

print(f"\n=== Summary ===")
print(f"Processed: {processed}")
print(f"Skipped (cached): {skipped}")
print(f"Failed: {failed}")
print(f"Total: {len(manifest)}")

## Cell 5 — Verify Cache

In [ ]:
import numpy as np
from pathlib import Path

print("=== Cache Verification ===\n")
tensors_dir = DRIVE_CACHE / "tensors"

for entry in manifest:
    tensor_dir = tensors_dir / entry["content_hash"]
    cortical_path = tensor_dir / "raw_cortical.npy"
    subcortical_path = tensor_dir / "raw_subcortical.npy"

    if cortical_path.exists():
        cortical = np.load(cortical_path)
        subcortical = np.load(subcortical_path) if subcortical_path.exists() else None
        sub_shape = subcortical.shape if subcortical is not None else "missing"
        print(f"\u2713 {entry['stimulus_id']:30s}  cortical={cortical.shape}  subcortical={sub_shape}")
    else:
        print(f"\u2717 {entry['stimulus_id']:30s}  NOT CACHED")